In [23]:
import sqlite3

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# --------------------------------------------------
# Load data from SQLite
# --------------------------------------------------

db_path = r"C:\Users\rrmel\Downloads\Fraud Data\fraud.db"

conn = sqlite3.connect(db_path)

df = pd.read_sql_query(
    "SELECT * FROM fraud",
    conn
)

conn.close()

db_path = r"C:\Users\rrmel\Downloads\Fraud Data\fraud.db"


# Clean bad database row
with sqlite3.connect(db_path) as conn:

    cursor = conn.cursor()

    cursor.execute("""
        DELETE FROM fraud
        WHERE is_fraud = 'is_fraud'
    """)

    conn.commit()


# Load cleaned database
with sqlite3.connect(db_path) as conn:

    df = pd.read_sql_query(
        "SELECT * FROM fraud",
        conn
    )


# Clean missing values
df = df.replace(
    ["", "''", '""'],
    np.nan
)


# Convert target
df["is_fraud"] = pd.to_numeric(
    df["is_fraud"],
    errors="coerce"
)

df = df.dropna(
    subset=["is_fraud"]
)

df["is_fraud"] = df["is_fraud"].astype(int)


print(df["is_fraud"].value_counts())
# --------------------------------------------------
# Define feature groups
# --------------------------------------------------

numeric_features = [
    "transaction_amount",
    "num_items",
    "customer_age",
    "prev_transactions",
    "distance_from_home",
    "network_quality",
    "velocity_score"
]

categorical_features = [
    "hour_of_day",
    "device_type",
    "store_type"
]

binary_features = [
    "is_weekend",
    "is_first_transaction"
]


# --------------------------------------------------
# Convert numeric columns
# --------------------------------------------------

for column in numeric_features:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# --------------------------------------------------
# Convert binary columns
# --------------------------------------------------

for column in binary_features:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

print("\nOverall fraud rate:")
print(df["is_fraud"].mean())

print("\nNumeric averages by fraud status:")
print(
    df.groupby("is_fraud")[numeric_features].mean()
)

for column in [
    "hour_of_day",
    "device_type",
    "store_type",
    "is_weekend",
    "is_first_transaction"
]:
    print(f"\nFraud rate by {column}:")

    print(
        df.groupby(column)["is_fraud"]
        .agg(["mean", "count"])
        .sort_values("mean", ascending=False)
    )

is_fraud
0    12558
1     1442
Name: count, dtype: int64

Overall fraud rate:
0.103

Numeric averages by fraud status:
          transaction_amount  num_items  customer_age  prev_transactions  \
is_fraud                                                                   
0                  99.807591   2.990301     36.252684           4.448400   
1                 101.818822   2.865629     35.754965           4.538231   

          distance_from_home  network_quality  velocity_score  
is_fraud                                                       
0                  24.441383        74.099972        5.023614  
1                  28.135538        74.049466        4.870711  

Fraud rate by hour_of_day:
                 mean  count
hour_of_day                 
1            0.117706   1988
3            0.100858   4660
2            0.100120   6652

Fraud rate by device_type:
                 mean  count
device_type                 
0            0.115132   6688
1            0.095817   4112
2  

In [ ]:
# --------------------------------------------------
# Create X and y
# --------------------------------------------------

X = df.drop(columns="is_fraud")
y = df["is_fraud"]


# --------------------------------------------------
# Inspect target distribution
# --------------------------------------------------

print("Dataset shape:")
print(X.shape)

print("\nFraud counts:")
print(y.value_counts())

print("\nFraud percentages:")
print(y.value_counts(normalize=True))


# --------------------------------------------------
# Numeric preprocessing
# --------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# --------------------------------------------------
# Categorical preprocessing
# --------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


# --------------------------------------------------
# Binary preprocessing
# --------------------------------------------------

binary_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        )
    ]
)


# --------------------------------------------------
# Combine preprocessing
# --------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        ),
        (
            "binary",
            binary_pipeline,
            binary_features
        )
    ]
)



Dataset shape:
(14000, 12)

Fraud counts:
is_fraud
0    12558
1     1442
Name: count, dtype: int64

Fraud percentages:
is_fraud
0    0.897
1    0.103
Name: proportion, dtype: float64


In [12]:
# --------------------------------------------------
# Create model pipeline
# --------------------------------------------------

model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )
    ]
)


# --------------------------------------------------
# Train/test split
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=123,
    stratify=y
)


# --------------------------------------------------
# Train model
# --------------------------------------------------

model.fit(
    X_train,
    y_train
)


# --------------------------------------------------
# Predictions
# --------------------------------------------------

predictions = model.predict(X_test)

prediction_probabilities = model.predict_proba(
    X_test
)[:, 1]


# --------------------------------------------------
# Evaluate model
# --------------------------------------------------

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        predictions
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions
    )
)

print(
    "ROC AUC:",
    roc_auc_score(
        y_test,
        prediction_probabilities
    )
)

print(
    df.groupby("is_fraud")[
        [
            "transaction_amount",
            "num_items",
            "customer_age",
            "prev_transactions",
            "distance_from_home",
            "network_quality",
            "velocity_score"
        ]
    ].mean()
)

for column in [
    "hour_of_day",
    "device_type",
    "store_type",
    "is_weekend",
    "is_first_transaction"
]:
    print(f"\nFraud rate by {column}:")

    print(
        df.groupby(column)["is_fraud"]
        .mean()
        .sort_values(ascending=False)
    )

print(
    "Overall fraud rate:",
    df["is_fraud"].mean()
)


Confusion Matrix:
[[1457 1055]
 [ 150  138]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.58      0.71      2512
           1       0.12      0.48      0.19       288

    accuracy                           0.57      2800
   macro avg       0.51      0.53      0.45      2800
weighted avg       0.83      0.57      0.65      2800

ROC AUC: 0.5570594479830149
          transaction_amount  num_items  customer_age  prev_transactions  \
is_fraud                                                                   
0                  99.807591   2.990301     36.252684           4.448400   
1                 101.818822   2.865629     35.754965           4.538231   
is_fraud                 NaN        NaN           NaN                NaN   

          distance_from_home  network_quality  velocity_score  
is_fraud                                                       
0                  24.441383        74.099972        5.023614  
1 

TypeError: agg function failed [how->mean,dtype->object]

In [25]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=123,
                n_jobs=-1
            )
        )
    ]
)

random_forest_model.fit(
    X_train,
    y_train
)

rf_predictions = random_forest_model.predict(
    X_test
)

rf_probabilities = random_forest_model.predict_proba(
    X_test
)[:, 1]

print("\nRandom Forest Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        rf_predictions
    )
)

print("\nRandom Forest Classification Report:")
print(
    classification_report(
        y_test,
        rf_predictions
    )
)

print(
    "Random Forest ROC AUC:",
    roc_auc_score(
        y_test,
        rf_probabilities
    )
)

feature_names = random_forest_model.named_steps[
    "preprocessor"
].get_feature_names_out()

importances = random_forest_model.named_steps[
    "classifier"
].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(
    "importance",
    ascending=False
)

print(importance_df.head(20))


Random Forest Confusion Matrix:
[[2512    0]
 [  60  228]]

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      2512
           1       1.00      0.79      0.88       288

    accuracy                           0.98      2800
   macro avg       0.99      0.90      0.94      2800
weighted avg       0.98      0.98      0.98      2800

Random Forest ROC AUC: 0.962650112791932
                         feature  importance
4    numeric__distance_from_home    0.150274
0    numeric__transaction_amount    0.148371
6        numeric__velocity_score    0.142692
5       numeric__network_quality    0.139229
2          numeric__customer_age    0.133133
3     numeric__prev_transactions    0.090480
1             numeric__num_items    0.067186
8     categorical__hour_of_day_2    0.015016
9     categorical__hour_of_day_3    0.014573
13     categorical__store_type_0    0.013147
15            binary__is_weekend    0.0

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Fraud Precision": [
        0.12,
        1.00
    ],
    "Fraud Recall": [
        0.48,
        0.79
    ],
    "Fraud F1": [
        0.19,
        0.88
    ],
    "ROC AUC": [
        0.557,
        0.963
    ]
})

print(results)


                 Model  Fraud Precision  Fraud Recall  Fraud F1  ROC AUC
0  Logistic Regression             0.12          0.48      0.19    0.557
1        Random Forest             1.00          0.79      0.88    0.963


In [27]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = cross_validate(
    random_forest_model,
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

for metric in scoring:
    scores = cv_results[f"test_{metric}"]

    print(
        f"{metric}: "
        f"{scores.mean():.3f} "
        f"(+/- {scores.std():.3f})"
    )

precision: 1.000 (+/- 0.000)
recall: 0.784 (+/- 0.025)
f1: 0.878 (+/- 0.016)
roc_auc: 0.964 (+/- 0.005)


In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    random_forest_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=123,
    scoring="roc_auc",
    n_jobs=-1
)

permutation_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
}).sort_values(
    "importance",
    ascending=False
)

print(permutation_df)